## 🕵️‍♂️ Case File #9: The Islamabad Telecom Churn & Customer Health Ledger

In [91]:
# import the necessary libraries used in the project
import pandas as pd
import numpy as np

# the dirty customer health dataset
dirty_telecom_data = {
    "Customer_ID": [501, 502, 503, 504, 505, 506, 507, 508],
    "Package": [" Premium ", "Basic", "Premium", "Standard", " Premium", "Basic", "Standard", "Premium"],
    "Monthly_Bill_PKR": ["5500", "2200", "5500", "3800", "5500", "2200", "3800", "5500"],
    "Data_Usage_GB": [450.5, 120.0, np.nan, 310.2, 500.0, np.nan, 280.5, 490.1],
    "Support_Tickets": [5, 1, 6, 2, 0, 8, 1, 7],
    "Churn_Risk": ["Yes", "No", "Yes", "No", "No", "Yes", "No", "Yes"]
}

# convert the list of dictionaries into a pandas dataframe to perform vectorized operations on the data
df_telecom = pd.DataFrame(dirty_telecom_data)
df_telecom.head(3)

,Customer_ID,Package,Monthly_Bill_PKR,Data_Usage_GB,Support_Tickets,Churn_Risk
0,501,Premium,5500,450.5,5,Yes
1,502,Basic,2200,120.0,1,No
2,503,Premium,5500,NaN,6,Yes


In [92]:
# get the over view of the dataset
df_telecom.info()

<class 'pandas.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Customer_ID       8 non-null      int64  
 1   Package           8 non-null      str    
 2   Monthly_Bill_PKR  8 non-null      str    
 3   Data_Usage_GB     6 non-null      float64
 4   Support_Tickets   8 non-null      int64  
 5   Churn_Risk        8 non-null      str    
dtypes: float64(1), int64(2), str(3)
memory usage: 516.0 bytes


#### 1. **Type Cast Correction:** The `Monthly_Bill_PKR` column accidentally loaded as a string data type. Clean it and permanently cast it into a proper numeric data type (`int` or `float`).

In [93]:
# convert the entire column values into float using the .astype() method
df_telecom["Monthly_Bill_PKR"] = df_telecom["Monthly_Bill_PKR"].astype(float)

# verify the results
df_telecom["Monthly_Bill_PKR"].dtype

dtype('float64')

#### 2. **Structural Text Alignment:** Look closely at the **"Package"** column. Extra whitespace spaces are padding the strings (e.g., `" Premium "` vs `"Premium"`). Standardize this column completely by removing all leading and trailing whitespaces across the whole series simultaneously.

In [94]:
df_telecom["Package"] = df_telecom["Package"].str.strip()

#### 3. **Impute Missing Metrics:** The `Data_Usage_GB` column contains missing `NaN` values. Do not drop these rows! Replace all missing values in that column with the overall mean data usage calculated from the rest of the valid users.

In [95]:
# replace the missing values with mean of the overall column values using fillna method
df_telecom.fillna(df_telecom["Data_Usage_GB"].mean(), inplace=True)

,Customer_ID,Package,Monthly_Bill_PKR,Data_Usage_GB,Support_Tickets,Churn_Risk
0,501,Premium,5500.0,450.50,5,Yes
1,502,Basic,2200.0,120.00,1,No
2,503,Premium,5500.0,358.55,6,Yes
3,504,Standard,3800.0,310.20,2,No
4,505,Premium,5500.0,500.00,0,No
5,506,Basic,2200.0,358.55,8,Yes
6,507,Standard,3800.0,280.50,1,No
7,508,Premium,5500.0,490.10,7,Yes


#### 4. **Isolate the Priority Risk Targets:** Filter the entire dataset down to show only high-value risk accounts. A customer is considered a Priority Risk Target if they meet all of the following conditions concurrently:

 - Their `Package` is exactly `"Premium"` AND

 - Their ``Support_Tickets`` count is strictly **greater than 4 AND**

 - Their `Churn_Risk` is marked as `"Yes"`.

In [96]:
# apply 3 conditions on multiple columns to figure out high value users at immediate risk of leaving the network
priority_risk_targets = df_telecom[
    (df_telecom["Package"] == "Premium") & 
    (df_telecom["Support_Tickets"] > 4) & 
    (df_telecom["Churn_Risk"] == "Yes")
    ]

# verify the results
priority_risk_targets

,Customer_ID,Package,Monthly_Bill_PKR,Data_Usage_GB,Support_Tickets,Churn_Risk
0,501,Premium,5500.0,450.50,5,Yes
2,503,Premium,5500.0,358.55,6,Yes
7,508,Premium,5500.0,490.10,7,Yes
